# 2. Datenbereinigung - Titanic Dataset

In diesem Notebook werden die Titanic-Daten für statistische Analysen und Machine-Learning-Modelle vorbereitet.

## Ziele
- Datensatz laden
- Fehlende Werte behandeln
- Unnötige oder problematische Spalten entfernen
- Kategorische Variablen kodieren
- Bereinigte Daten speichern

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
print('Bibliotheken erfolgreich importiert!')

## 2.1 Datensatz laden

In [ ]:
data_path = Path('../data/raw/titanic.csv')
df = pd.read_csv(data_path)

print(f'Dataset geladen: {data_path}')
print(f'Zeilen: {df.shape[0]} | Spalten: {df.shape[1]}')
df.head()

## 2.2 Fehlende Werte und Spalten prüfen

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'fehlende_werte': missing,
    'prozent': missing_percent.round(2)
})

missing_summary[missing_summary['fehlende_werte'] > 0]

## 2.3 Bereinigungsstrategie

Ich verwende die folgenden Schritte:
- `deck` entfernen, da sehr viele Werte fehlen
- `alive` entfernen, da diese Spalte die Zielvariable in Textform dupliziert
- `class` entfernen, da sie inhaltlich stark mit `pclass` überlappt
- Fehlende Werte in `age` mit dem Median ersetzen
- Fehlende Werte in `embarked` und `embark_town` mit dem Modus ersetzen

In [ ]:
df_clean = df.copy()

columns_to_drop = [col for col in ['deck', 'alive', 'class'] if col in df_clean.columns]
df_clean = df_clean.drop(columns=columns_to_drop)

if 'age' in df_clean.columns:
    df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())

for col in ['embarked', 'embark_town']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print('Bereinigung abgeschlossen.')
print(f'Neue Form: {df_clean.shape}')
df_clean.isnull().sum().sort_values(ascending=False).head(10)

## 2.4 Kategorische Variablen kodieren

Für Machine-Learning-Modelle werden kategoriale Spalten in numerische Form überführt.

In [ ]:
categorical_columns = df_clean.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print('Kategoriale Spalten:', categorical_columns)

df_model = pd.get_dummies(df_clean, columns=categorical_columns, drop_first=True)
print(f'Modelldaten Form: {df_model.shape}')
df_model.head()

## 2.5 Bereinigte Daten speichern

In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

clean_output = processed_dir / 'titanic_cleaned.csv'
model_output = processed_dir / 'titanic_model_ready.csv'

df_clean.to_csv(clean_output, index=False)
df_model.to_csv(model_output, index=False)

print(f'Gespeichert: {clean_output}')
print(f'Gespeichert: {model_output}')